# K-Nearest Neighbors (KNN) Regression: Finding Similar Players

**Chapter 6: Regression Techniques for Soccer Analytics**

## Learning Objectives

- Understand how KNN regression works
- Learn the difference between parametric and non-parametric models
- Build a KNN model to predict player values
- Choose the optimal value of K
- Visualize decision boundaries
- Compare KNN with linear regression


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


## The Intuition: Birds of a Feather

**KNN Regression** is fundamentally different from linear or Poisson regression:

- **Linear/Poisson:** Learn a formula from all data, then apply it
- **KNN:** For each prediction, find the K most similar data points and average their values

**Example:** To predict a player's market value:
1. Find the 5 most similar players (based on goals, assists, age, etc.)
2. Average their market values
3. That's your prediction!

**No formula, no training in the traditional sense** — just similarity-based lookup!


## Parametric vs. Non-Parametric

| Aspect | Parametric (Linear, Poisson) | Non-Parametric (KNN) |
|--------|------------------------------|----------------------|
| **Model** | Learns fixed parameters (slope, intercept) | No fixed parameters |
| **Training** | Fits equation to data | Memorizes all data |
| **Prediction** | Applies learned formula | Finds similar points |
| **Flexibility** | Assumes specific relationship | Adapts to any pattern |
| **Interpretability** | High (clear coefficients) | Low (black box) |


## Load Data: Player Statistics

We'll use a richer dataset with multiple features.


In [ ]:
# Simulated player data with multiple features
np.random.seed(42)
n_players = 50

data = {
    'Goals': np.random.randint(5, 30, n_players),
    'Assists': np.random.randint(2, 15, n_players),
    'Age': np.random.randint(20, 35, n_players),
    'MarketValue_Millions': np.random.randint(10, 100, n_players)
}

df = pd.DataFrame(data)
# Add some correlation
df['MarketValue_Millions'] = (df['Goals'] * 2.5 + df['Assists'] * 1.5 - df['Age'] * 0.5 + 
                               np.random.normal(0, 10, n_players)).clip(10, 100)

print(\"Player Dataset:\")
print(df.head(10))
print(f\"\
Dataset shape: {df.shape}\")
print(f\"\
Feature correlations with Market Value:\")
print(df.corr()['MarketValue_Millions'].sort_values(ascending=False))


## Prepare Data

KNN is **distance-based**, so feature scaling is crucial!


In [ ]:
# Select features and target
features = ['Goals', 'Assists', 'Age']
target = 'MarketValue_Millions'

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (important for KNN!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f\"Training set: {len(X_train)} players\")
print(f\"Test set: {len(X_test)} players\")
print(f\"\
Features are now scaled to have mean=0 and std=1\")
print(\"This ensures all features contribute equally to distance calculations.\")


## Build KNN Model

Let's start with K=5 (using 5 nearest neighbors).


In [ ]:
# Create and train KNN model
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Make predictions
y_pred_train = knn.predict(X_train_scaled)
y_pred_test = knn.predict(X_test_scaled)

# Evaluate
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(\"KNN Model (K=5) Performance:\")
print(f\"  Training R²: {train_r2:.3f}\")
print(f\"  Test R²:     {test_r2:.3f}\")
print(f\"  Training RMSE: €{train_rmse:.2f}M\")
print(f\"  Test RMSE:     €{test_rmse:.2f}M\")


## Choosing the Right K

**K** is a hyperparameter that controls model complexity:

- **K = 1:** Very flexible, prone to overfitting (uses only the closest neighbor)
- **K = large:** Very smooth, prone to underfitting (averages many neighbors)
- **Optimal K:** Balance between bias and variance

Let's find the best K using cross-validation.


In [ ]:
# Test different K values
k_values = range(1, 21)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(k_values, train_scores, label='Training R²', marker='o')
plt.plot(k_values, test_scores, label='Test R²', marker='s')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('R² Score')
plt.title('KNN Performance vs. K Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find best K
best_k = k_values[np.argmax(test_scores)]
print(f\"\
Best K value: {best_k}\")
print(f\"Best test R²: {max(test_scores):.3f}\")
print(f\"\
Observation:\")
print(f\"- Low K: High training score, lower test score (overfitting)\")
print(f\"- High K: Both scores converge (underfitting)\")


## Visualize KNN Predictions

Let's visualize predictions for a 2D slice of the data.


In [ ]:
# Train KNN with best K on 2 features for visualization
X_2d = df[['Goals', 'Assists']]
y_2d = df['MarketValue_Millions']

scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

knn_2d = KNeighborsRegressor(n_neighbors=best_k)
knn_2d.fit(X_2d_scaled, y_2d)

# Create prediction grid
goals_range = np.linspace(X_2d['Goals'].min(), X_2d['Goals'].max(), 50)
assists_range = np.linspace(X_2d['Assists'].min(), X_2d['Assists'].max(), 50)
goals_grid, assists_grid = np.meshgrid(goals_range, assists_range)

grid_points = np.c_[goals_grid.ravel(), assists_grid.ravel()]
grid_scaled = scaler_2d.transform(grid_points)
predictions = knn_2d.predict(grid_scaled).reshape(goals_grid.shape)

# Plot
plt.figure(figsize=(12, 8))
contour = plt.contourf(goals_grid, assists_grid, predictions, levels=15, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Predicted Market Value (€M)')
plt.scatter(df['Goals'], df['Assists'], c=df['MarketValue_Millions'], 
            s=100, edgecolors='black', cmap='viridis', label='Actual Players')
plt.xlabel('Goals')
plt.ylabel('Assists')
plt.title(f'KNN Decision Surface (K={best_k})')
plt.legend()
plt.tight_layout()
plt.show()

print(\"The contour plot shows how KNN creates prediction regions.\")
print(\"Notice the non-linear, flexible boundaries!\")


## Compare KNN vs. Linear Regression


In [ ]:
from sklearn.linear_model import LinearRegression

# Train linear model
linear = LinearRegression()
linear.fit(X_train_scaled, y_train)

# Compare performance
knn_final = KNeighborsRegressor(n_neighbors=best_k)
knn_final.fit(X_train_scaled, y_train)

linear_test_r2 = linear.score(X_test_scaled, y_test)
knn_test_r2 = knn_final.score(X_test_scaled, y_test)

linear_test_rmse = np.sqrt(mean_squared_error(y_test, linear.predict(X_test_scaled)))
knn_test_rmse = np.sqrt(mean_squared_error(y_test, knn_final.predict(X_test_scaled)))

print(\"Model Comparison on Test Set:\")
print(f\"\
Linear Regression:\")
print(f\"  R²: {linear_test_r2:.3f}\")
print(f\"  RMSE: €{linear_test_rmse:.2f}M\")
print(f\"\
KNN (K={best_k}):\")
print(f\"  R²: {knn_test_r2:.3f}\")
print(f\"  RMSE: €{knn_test_rmse:.2f}M\")

if knn_test_r2 > linear_test_r2:
    print(\"\
KNN performs better! It can capture non-linear patterns.\")
else:
    print(\"\
Linear regression performs better! The relationship may be mostly linear.\")


## Find Similar Players

One powerful use of KNN: finding similar players!


In [ ]:
# Find similar players to a target player
target_idx = 0
target_player = X_train.iloc[target_idx]
target_scaled = X_train_scaled[target_idx].reshape(1, -1)

# Find K nearest neighbors
knn_finder = KNeighborsRegressor(n_neighbors=5)
knn_finder.fit(X_train_scaled, y_train)
distances, indices = knn_finder.kneighbors(target_scaled)

print(\"Target Player:\")
print(target_player)
print(f\"Market Value: €{y_train.iloc[target_idx]:.1f}M\")
print(f\"\
5 Most Similar Players:\")

for i, (dist, idx) in enumerate(zip(distances[0][1:], indices[0][1:]), 1):
    similar_player = X_train.iloc[idx]
    similar_value = y_train.iloc[idx]
    print(f\"\
{i}. Distance: {dist:.3f}\")
    print(f\"   Goals: {similar_player['Goals']}, Assists: {similar_player['Assists']}, Age: {similar_player['Age']}\")
    print(f\"   Market Value: €{similar_value:.1f}M\")


## Advantages and Disadvantages of KNN

### Advantages
| Advantage | Description |
|-----------|-------------|
| **No assumptions** | Works with any data distribution |
| **Flexible** | Captures complex non-linear patterns |
| **Simple concept** | Easy to understand intuitively |
| **Multi-output** | Can predict multiple targets simultaneously |

### Disadvantages
| Disadvantage | Description |
|--------------|-------------|
| **Computationally expensive** | Must search all training data for each prediction |
| **Requires scaling** | Sensitive to feature scales |
| **Curse of dimensionality** | Performance degrades with many features |
| **Not interpretable** | Can't explain predictions with coefficients |


## Summary

In this notebook, we:

1. ✅ Understood how KNN regression works (similarity-based prediction)
2. ✅ Learned the difference between parametric and non-parametric models
3. ✅ Built a KNN model to predict player values
4. ✅ Found the optimal K value using cross-validation
5. ✅ Visualized KNN decision boundaries
6. ✅ Compared KNN with linear regression
7. ✅ Used KNN to find similar players

## Key Takeaways

- **KNN** makes predictions by averaging K nearest neighbors
- **Non-parametric** → no fixed formula, very flexible
- **Feature scaling is crucial** for distance-based methods
- **K is a hyperparameter** that controls model complexity
- **Great for finding similar players** or situations
- **Trade-off:** Flexibility vs. interpretability and speed

## Next Steps

In the next notebook, we'll learn comprehensive **Model Evaluation and Diagnostics** techniques!


## Practice Exercises

1. **Distance Metrics**: Try different distance metrics (Manhattan, Minkowski)
2. **Weighted KNN**: Use distance-weighted predictions
3. **More Features**: Add more player statistics and see how it affects performance
4. **Player Recommendation System**: Build a system to recommend similar players for scouting
5. **Curse of Dimensionality**: Experiment with 10+ features and observe performance degradation
